In [2]:
!pip install -q -U pydantic-ai "pydantic-ai-slim[groq]"

In [19]:
# because of python is doing the async tasks here so that why need to include it
import nest_asyncio

nest_asyncio.apply()

In [20]:
import requests

In [17]:
import requests

BASE_URL = "https://api.openweathermap.org/data/2.5/weather"
# TODO: Move this to an environment variable in production!
API_KEY = ""

def find_weather(city: str) -> dict:
    """This function returns the current weather forecast for the city"""

    params = {
        'q': city,
        'appid': API_KEY,
        'units': 'metric'  # Fixed: changed 'metrics' to 'metric'
    }

    try:
        response = requests.get(BASE_URL, params=params)

        # This will raise an exception if the API returns a 404 (City Not Found) or 401 (Invalid API Key)
        response.raise_for_status()

        return response.json()

    except requests.exceptions.HTTPError as err:
        print(f"HTTP Error: {err} - Could not find weather for '{city}'")
        return {}
    except requests.exceptions.RequestException as err:
        print(f"Connection Error: {err}")
        return {}

In [ ]:
import requests

# Your API key and city are hardcoded correctly below
API_KEY = ""
city = "Ahmedabad"

# Corrected Free plan URL structure
url = f"https://openweathermap.org{city}&appid={API_KEY}"

try:
    response = requests.get(url)

    if response.status_code == 200:
        print("✅ SUCCESS! Your API key is active and working.")
        # Extracted the description safely from the list
        weather_desc = response.json()['weather'][0]['description']
        print(f"Current weather in {city}: {weather_desc}")
    elif response.status_code == 401:
        print("❌ FAILED: 401 Unauthorized.")
        print("Your key is either still activating, or it was copied incorrectly.")
    else:
        print(f"❌ ERROR: Status code {response.status_code}")
        print(response.json())

except Exception as e:
    print(f"Network error: {e}")


Network error: HTTPSConnectionPool(host='openweathermap.orgahmedabad&appid=70dd4296422d86e61dee21338b4d27b8', port=443): Max retries exceeded with url: / (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7933eab938c0>: Failed to resolve 'openweathermap.orgahmedabad&appid=70dd4296422d86e61dee21338b4d27b8' ([Errno -2] Name or service not known)"))


In [15]:
import requests

# This URL is completely hardcoded to prevent any string formatting bugs
url = "https://openweathermap.org"

try:
    response = requests.get(url)

    if response.status_code == 200:
        print("✅ SUCCESS! Your API key is active and working.")
        # Accessing the first element [0] of the weather list
        weather_desc = response.json()['weather'][0]['description']
        print(f"Current weather in Ahmedabad: {weather_desc}")
    elif response.status_code == 401:
        print("❌ FAILED: 401 Unauthorized.")
        print("The URL is correct now, but your API key is still activating. Wait a bit longer.")
    else:
        print(f"❌ ERROR: Status code {response.status_code}")
        print(response.json())

except Exception as e:
    print(f"Network error: {e}")


✅ SUCCESS! Your API key is active and working.
Network error: Expecting value: line 1 column 1 (char 0)


In [18]:
output = find_weather('London')
print(output)

{'coord': {'lon': -0.1257, 'lat': 51.5085}, 'weather': [{'id': 804, 'main': 'Clouds', 'description': 'overcast clouds', 'icon': '04d'}], 'base': 'stations', 'main': {'temp': 14.14, 'feels_like': 13.7, 'temp_min': 13.34, 'temp_max': 14.49, 'pressure': 995, 'humidity': 80, 'sea_level': 995, 'grnd_level': 991}, 'visibility': 10000, 'wind': {'speed': 3.13, 'deg': 240, 'gust': 7.15}, 'clouds': {'all': 99}, 'dt': 1780551539, 'sys': {'type': 2, 'id': 2075535, 'country': 'GB', 'sunrise': 1780544826, 'sunset': 1780603849}, 'timezone': 3600, 'id': 2643743, 'name': 'London', 'cod': 200}


In [22]:
print(output['name'])
print(output['weather'][0]["description"].capitalize())
print(output['main']['temp'])

London
Overcast clouds
14.14


In [23]:
import os
import requests
from pydantic import BaseModel
from pydantic_ai import Agent, RunContext
from pydantic_ai.settings import ModelSettings

In [24]:
os.environ['GROQ_API_KEY'] = ""

In [25]:
# define the output schema for the tool

class WeatherForecast(BaseModel):
  location: str
  description: str
  temperature_celsius: float

In [27]:
weather_agent = Agent(
    model="groq:llama-3.1-8b-instant",
    model_settings=ModelSettings(temperature=0.1),
    output_type=str,
    system_prompt=("You are a helpful weather assistant.Use the 'get_weather_forecast' tool"
                    "to find current weather conditions for any city.Provide clean and friendly answers"
                  )
)

In [28]:
# weather forecast tool
@weather_agent.tool
def get_weather_forecast(ctx: RunContext,city:str) -> WeatherForecast:
  """
  Returns weather forecast for a city using OpenWeather API.
  """
  url = "https://api.openweathermap.org/data/2.5/weather"
  params = {
      'q': city,
      'appid':API_KEY,
      'units': 'metric'
  }
  response = requests.get(url,params=params)
  data = response.json()

  return WeatherForecast(
      location=data['name'],
      description=data['weather'][0]['description'].capitalize(),
      temperature_celsius=data['main']['temp']
  )

In [29]:
agent_response = weather_agent.run_sync("What is the weather of Dabhoi")
print(agent_response.output)

The current weather in Dabhoi is clear sky with a temperature of 37.47°C.


In [35]:
question = input("🌤️ Ask about the weather: ")
result = weather_agent.run_sync(question)
print("\n📍 Forecast: ",result.output)

🌤️ Ask about the weather: Ahmedabad

📍 Forecast:  It's currently 39.84°C in Ahmedabad with a clear sky.


In [ ]:
""" this is the script that we can make to run it properly
Weather Assistant using Pydantic-AI and OpenWeatherMap API
This script creates a conversational agent that can respond to weather-related queries
using the OpenWeatherMap API and a Groq-hosted LLaMA model.
"""

# Allow nested async loops (useful for Jupyter or async environments)
import nest_asyncio
nest_asyncio.apply()

# Standard libraries
import os
import requests

# Pydantic model for structured data
from pydantic import BaseModel

# Core AI libraries from pydantic_ai
from pydantic_ai import Agent, RunContext
from pydantic_ai.settings import ModelSettings

os.environ["GROQ_API_KEY"] = ""

# 1. Define the output schema of the tool using Pydantic
class WeatherForecast(BaseModel):
    location: str
    description: str
    temperature_celsius: float

# 2. Create the AI agent using Groq’s LLaMA 3 model
weather_agent = Agent(
    model="groq:llama-3.1-8b-instant",
    model_settings=ModelSettings(temperature=0.2),
    output_type=str,
    system_prompt=(
        "You are a helpful weather assistant. Use the 'get_weather_forecast' tool to "
        "find current weather conditions for any city. Provide clean and friendly answers."
    )
)

# 3. Register a tool with the agent to fetch real-time weather using OpenWeatherMap API
@weather_agent.tool
def get_weather_forecast(ctx: RunContext, city: str) -> WeatherForecast:
    """
    Tool: get_weather_forecast
    Description: Fetches current weather for a city using the OpenWeatherMap API.
    """
    url = "https://api.openweathermap.org/data/2.5/weather"

    # Replace this with your own API key for production use
    api_key = ""

    # Query parameters
    params = {
        'q': city,
        'appid': api_key,
        'units': 'metric'
    }

    # Send request to weather API
    res = requests.get(url, params=params).json()

    # Return the formatted weather information
    return WeatherForecast(
        location=res["name"],
        description=res["weather"][0]["description"].capitalize(),
        temperature_celsius=res["main"]["temp"]
    )

# 4. Run continuous user interaction loop
if __name__ == "__main__":
    print("🌦️  Weather Assistant is ready! Type 'exit' to quit.")
    print("‒" * 50)
    while True:
        question = input("🌤️ Ask about the weather: ").strip()
        if question.lower() in {"exit", "quit", ""}:
            print("\n👋 Exiting weather assistant. Have a nice day!")
            break

        try:
            result = weather_agent.run_sync(question)
            print("\n📍 Forecast:", result.output)
        except Exception as e:
            print("⚠️ Error:", str(e))

        print("‒" * 50)

🌦️  Weather Assistant is ready! Type 'exit' to quit.
‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒
🌤️ Ask about the weather: London

📍 Forecast: It's overcast in London with a temperature of 14.36°C.
‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒
🌤️ Ask about the weather: Paris

📍 Forecast: <function=get_weather_forecast>{"city": "Paris"}
‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒
🌤️ Ask about the weather: Indore

📍 Forecast: <function=get_weather_forecast>{"city": "Indore"}
‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒
🌤️ Ask about the weather: Vadodara

📍 Forecast: It seems like the weather in Vadodara is clear with a temperature of 37.09°C.
‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒
🌤️ Ask about the weather: Surat

📍 Forecast: <function=get_weather_forecast>{"city": "Surat"}
‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒
🌤️ Ask about the weather: Ahmedabad

📍 Forecast: <function=get_weather_forecast>{"city": "Ahmedabad"}
‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒

KeyboardInterrupt: Interrupted by user